# Satellite Orbit Explorer · built during Space Career Week 2026 · GNSS Academy

Eight stages:
1. Imports and configuration
2. Load GP JSON
3. Parse orbital elements
4. Time window
5. Propagate
6. Orbit summary
7. Plots
8. KML and report

**Model and limitations:** Educational two-body Kepler orbit, spherical Earth and simple Greenwich rotation. GP records provide SGP4 mean elements, so predictions using this simplified model are approximations, **not precision orbit predictions**.

All helper functions will be defined inside notebook cells (no external `.py` modules).

In [ ]:
import json
import math
import time
import zipfile
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta, timezone
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
import requests

# Educational two-body orbital model constants (km, s).
MU_EARTH_KM3_S2 = 398600.4418
EARTH_RADIUS_KM = 6378.137

# Colab input/output directories.
BASE_DIR = Path("/content/satellite_orbits")
IN_DIR = BASE_DIR / "in"
OUT_DIR = BASE_DIR / "out"
IN_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Stage 1 configuration. LIVE_GROUP is reserved for an optional later extension.
STEP_SECONDS = 5
DURATION_HOURS = 24
LIVE_GROUP = None

print("Satellite Orbit Explorer — configuration")
print(f"Input directory:  {IN_DIR}")
print(f"Output directory: {OUT_DIR}")
print(f"Time window:      {DURATION_HOURS} hours")
print(f"Sampling step:    {STEP_SECONDS} seconds")
print(f"Live group:       {LIVE_GROUP}")
print("Conventions: UTC-aware datetimes, km, km/s")


## Stage 2 — Load GP JSON

OMM JSON uses named fields rather than fixed-width TLE lines, making each orbital element easier to identify. It also supports ISO dates and six-digit catalog numbers without the old TLE format's limitations.

In [ ]:
# Use the three original workshop inputs; do not synthesize data.
INPUT_NAMES = ("orbital_elements.json", "data_manifest.json", "world_map.txt")

def extract_inputs():
    if all((IN_DIR / name).is_file() for name in INPUT_NAMES):
        return

    print("Drag orbit_workshop_inputs.zip into /content using the Colab Files panel.")
    deadline = time.monotonic() + 15 * 60
    archive = None
    while time.monotonic() < deadline:
        for candidate in sorted(Path("/content").glob("orbit_workshop_inputs*.zip")):
            if zipfile.is_zipfile(candidate):
                archive = candidate
                break
        if archive is not None:
            break
        time.sleep(2)
    if archive is None:
        raise TimeoutError("No completed orbit_workshop_inputs*.zip appeared in /content within 15 minutes.")

    with zipfile.ZipFile(archive) as zf:
        # Accept files at archive root or in a folder, without extracting other entries.
        entries = {}
        for info in zf.infolist():
            if not info.is_dir() and Path(info.filename).name in INPUT_NAMES:
                name = Path(info.filename).name
                if name in entries:
                    raise ValueError(f"Duplicate input in ZIP: {name}")
                entries[name] = info
        missing = set(INPUT_NAMES) - entries.keys()
        if missing:
            raise ValueError(f"Missing files in ZIP: {', '.join(sorted(missing))}")
        for name in INPUT_NAMES:
            # Read fully before writing: catches truncated archives and CRC errors.
            content = zf.read(entries[name])
            (IN_DIR / name).write_bytes(content)


def read_gp_json(path):
    with open(path, encoding="utf-8") as stream:
        records = json.load(stream)
    if not isinstance(records, list) or not records:
        raise ValueError("GP JSON must contain a non-empty list of records.")
    required = {
        "OBJECT_NAME", "NORAD_CAT_ID", "EPOCH", "MEAN_MOTION",
        "ECCENTRICITY", "INCLINATION", "RA_OF_ASC_NODE",
        "ARG_OF_PERICENTER", "MEAN_ANOMALY",
    }
    for index, record in enumerate(records, start=1):
        if not isinstance(record, dict):
            raise ValueError(f"GP record {index} is not a JSON object.")
        missing = required - record.keys()
        if missing:
            raise ValueError(f"GP record {index} missing fields: {', '.join(sorted(missing))}")
    return records


extract_inputs()
sats = read_gp_json(IN_DIR / "orbital_elements.json")
with open(IN_DIR / "data_manifest.json", encoding="utf-8") as stream:
    data_manifest = json.load(stream)
if not isinstance(data_manifest, dict) or not data_manifest.get("retrieved_utc"):
    raise ValueError("data_manifest.json must contain retrieved_utc.")
source_note = f"CelesTrak GP JSON (OMM fields), retrieved UTC: {data_manifest['retrieved_utc']}"
print(source_note)
for sat in sats:
    print(f"{sat['OBJECT_NAME']} | NORAD {sat['NORAD_CAT_ID']} | EPOCH {sat['EPOCH']}")


In [ ]:
# Optional live CelesTrak group: reuse the workshop snapshot unless enabled.
if not all(name in globals() for name in ("LIVE_GROUP", "IN_DIR", "sats", "source_note", "read_gp_json")):
    raise RuntimeError("Run Stages 1 and 2 before the live extension.")

if LIVE_GROUP is None:
    print("Using workshop snapshot; live CelesTrak download disabled.")
else:
    # Cache each group for two hours to respect CelesTrak's polling guidance.
    if not isinstance(LIVE_GROUP, str) or not LIVE_GROUP or not all(
        ch.isalnum() and ch.isascii() or ch in "_-" for ch in LIVE_GROUP
    ):
        raise ValueError("LIVE_GROUP must be a non-empty ASCII group name.")
    cache = IN_DIR / f"gp_{LIVE_GROUP}.json"
    if not cache.is_file() or time.time() - cache.stat().st_mtime > 2 * 3600:
        url = f"https://celestrak.org/NORAD/elements/gp.php?GROUP={LIVE_GROUP}&FORMAT=JSON"
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        records = response.json()
        if not isinstance(records, list) or not records:
            raise ValueError("CelesTrak returned no GP records.")
        # Validate before replacing any cached dataset.
        temp = cache.with_suffix(".tmp")
        try:
            temp.write_text(json.dumps(records), encoding="utf-8")
            read_gp_json(temp)
            temp.replace(cache)
        finally:
            temp.unlink(missing_ok=True)

    live_sats = read_gp_json(cache)
    if len(live_sats) < 4:
        raise ValueError(f"Group {LIVE_GROUP} contains fewer than four GP records.")
    sats = sorted(live_sats, key=lambda rec: int(rec["NORAD_CAT_ID"]))[:4]
    retrieved_utc = datetime.fromtimestamp(cache.stat().st_mtime, tz=timezone.utc)
    source_note = (
        f"CelesTrak GP JSON group {LIVE_GROUP}, retrieved UTC: "
        f"{retrieved_utc.strftime('%Y-%m-%dT%H:%M:%SZ')}"
    )
    print(source_note)
    for sat in sats:
        print(f"{sat['OBJECT_NAME']} | NORAD {sat['NORAD_CAT_ID']} | EPOCH {sat['EPOCH']}")


## Stage 3 — Convert GP records to Kepler elements

| GP field | Symbol | Conversion / unit |
|---|---|---|
| `OBJECT_NAME` | name | Text; unchanged |
| `NORAD_CAT_ID` | ID | Integer; unchanged |
| `EPOCH` | epoch | ISO UTC text → timezone-aware datetime |
| `MEAN_MOTION` | n | rev/day × 2π / 86400 → rad/s |
| `MEAN_MOTION` + Earth μ | a | (μ / n²)^(1/3) → km |
| `ECCENTRICITY` | e | Decimal; unchanged (dimensionless) |
| `INCLINATION` | i | degrees → radians |
| `RA_OF_ASC_NODE` | Ω (RAAN) | degrees → radians |
| `ARG_OF_PERICENTER` | ω | degrees → radians |
| `MEAN_ANOMALY` | M₀ | degrees → radians |

GP values are **SGP4 mean elements**; using them in our two-body Kepler model gives an educational approximation, not a precision orbit prediction.

In [ ]:
# Stage 3 relies on GP records loaded by Stage 2.
if "sats" not in globals() or "MU_EARTH_KM3_S2" not in globals():
    raise RuntimeError("Run Stages 1 and 2 before Stage 3.")

def parse_gp_epoch(text):
    # CelesTrak's offset-free GP epochs are in UTC.
    return datetime.fromisoformat(text.rstrip("Z")).replace(tzinfo=timezone.utc)


def parse_gp_elements(rec):
    name = rec["OBJECT_NAME"]
    norad_id = int(rec["NORAD_CAT_ID"])
    n = 2 * math.pi * float(rec["MEAN_MOTION"]) / 86400
    if n <= 0:
        raise ValueError(f"Non-positive mean motion for NORAD {norad_id}")
    return {
        "name": name,
        "norad_id": norad_id,
        "label": f"{name} [{norad_id}]",
        "epoch": parse_gp_epoch(rec["EPOCH"]),
        "n_rad_s": n,
        "a_km": (MU_EARTH_KM3_S2 / n**2)**(1 / 3),
        "e": float(rec["ECCENTRICITY"]),
        "i_rad": math.radians(float(rec["INCLINATION"])),
        "raan_rad": math.radians(float(rec["RA_OF_ASC_NODE"])),
        "argp_rad": math.radians(float(rec["ARG_OF_PERICENTER"])),
        "M0_rad": math.radians(float(rec["MEAN_ANOMALY"])),
    }


def elements_table(elements):
    # Return text lines for reuse by the Stage 8 report.
    lines = [
        "Name | NORAD | Epoch (UTC) | a [km] | e | i [deg] | RAAN [deg] | argp [deg] | M0 [deg]"
    ]
    for el in elements:
        lines.append(
            f"{el['name']} | {el['norad_id']} | {el['epoch'].isoformat()} | "
            f"{el['a_km']:.2f} | {el['e']:.8f} | "
            f"{math.degrees(el['i_rad']):.4f} | {math.degrees(el['raan_rad']):.4f} | "
            f"{math.degrees(el['argp_rad']):.4f} | {math.degrees(el['M0_rad']):.4f}"
        )
    return lines


elements = [parse_gp_elements(s) for s in sats]
print("\n".join(elements_table(elements)))


## Stage 4 — Common UTC time grid

Each satellite is propagated from its **own** GP epoch, so the farther the common start time is from that epoch, the larger the possible prediction error. The workshop's QZS-2 record dates from **17 October 2026**, making it a stale-data example.

In [ ]:
# Stage 4: one inclusive UTC grid for all satellite records.
if "elements" not in globals() or "DURATION_HOURS" not in globals() or "STEP_SECONDS" not in globals():
    raise RuntimeError("Run Stages 1–3 before Stage 4.")
if not elements:
    raise ValueError("Stage 3 returned no satellite elements.")
if STEP_SECONDS <= 0 or DURATION_HOURS <= 0:
    raise ValueError("DURATION_HOURS and STEP_SECONDS must be positive.")

newest_epoch = max(el["epoch"] for el in elements)
start = newest_epoch.astimezone(timezone.utc).replace(hour=0, minute=0, second=0, microsecond=0)
end = start + timedelta(hours=DURATION_HOURS)
n_points = int((end - start).total_seconds() // STEP_SECONDS) + 1
times = [start + timedelta(seconds=k * STEP_SECONDS) for k in range(n_points)]

print(f"UTC window: {start.isoformat()} to {end.isoformat()}")
print(f"Grid points: {n_points}")
for el in elements:
    offset_days = (start - el["epoch"]).total_seconds() / 86400
    status = " — STALE (>7 days)" if offset_days > 7 else ""
    print(f"{el['label']}: start - epoch = {offset_days:+.2f} days{status}")


## Stage 5 — Propagate orbits (two-body approximation)

1. Advance the mean anomaly from each satellite's own epoch to the requested UTC time.
2. Solve Kepler's equation numerically for eccentric anomaly `E`.
3. Convert `E` into orbital radius and true anomaly, then add the argument of pericenter.
4. Rotate the orbital position into the Earth-centered inertial frame using inclination and ascending-node longitude.
5. Use Greenwich mean sidereal time (IAU-1982) to rotate the inertial coordinates into Earth-fixed coordinates.
6. Convert the Earth-fixed position to spherical latitude, longitude and height above the spherical Earth.
7. Compute orbital speed from the vis-viva equation.

**Limit:** This Kepler model uses GP/SGP4 mean elements as approximations; it ignores perturbations and is not a precision ephemeris.

In [ ]:
# Stage 5 needs the orbital elements and UTC grid from earlier stages.
if not all(key in globals() for key in ("elements", "times", "MU_EARTH_KM3_S2", "EARTH_RADIUS_KM")):
    raise RuntimeError("Run Stages 1–4 before Stage 5.")
if not elements or not times:
    raise ValueError("Elements and time grid must be non-empty.")

def solve_kepler(M, e):
    if not (0 <= e < 1):
        raise ValueError("Elliptic eccentricity must satisfy 0 <= e < 1.")
    E = math.pi if e > 0.8 else M
    for _ in range(50):
        delta = (E - e * math.sin(E) - M) / (1 - e * math.cos(E))
        E -= delta
        if abs(E - e * math.sin(E) - M) < 1e-12:
            return E
    raise RuntimeError("Kepler solver did not converge in 50 iterations.")


def gmst_rad(t):
    j2000 = datetime(2000, 1, 1, 12, tzinfo=timezone.utc)
    d = (t - j2000).total_seconds() / 86400
    T = d / 36525
    theta = 280.46061837 + 360.98564736629 * d + 0.000387933 * T**2 - T**3 / 38710000
    return math.radians(theta % 360)


def propagate(el, times):
    a, e, i = el["a_km"], el["e"], el["i_rad"]
    raan, argp, n = el["raan_rad"], el["argp_rad"], el["n_rad_s"]
    result = []
    for t in times:
        # 1) Advance mean anomaly from this satellite's own epoch.
        M = (el["M0_rad"] + n * (t - el["epoch"]).total_seconds()) % (2 * math.pi)
        # 2) Solve Kepler's equation for eccentric anomaly.
        E = solve_kepler(M, e)
        # 3) Orbital radius, true anomaly, argument of latitude.
        r = a * (1 - e * math.cos(E))
        nu = math.atan2(math.sqrt(1 - e**2) * math.sin(E), math.cos(E) - e)
        u = argp + nu
        # 4) Rotate orbital coordinates into the inertial frame.
        x = r * (math.cos(raan)*math.cos(u) - math.sin(raan)*math.sin(u)*math.cos(i))
        y = r * (math.sin(raan)*math.cos(u) + math.cos(raan)*math.sin(u)*math.cos(i))
        z = r * math.sin(u)*math.sin(i)
        # 5) Greenwich sidereal rotation into Earth-fixed axes.
        theta = gmst_rad(t)
        xp = math.cos(theta)*x + math.sin(theta)*y
        yp = -math.sin(theta)*x + math.cos(theta)*y
        # 6) Spherical geodetic-like latitude, longitude and altitude.
        lat = math.degrees(math.atan2(z, math.hypot(xp, yp)))
        lon = math.degrees(math.atan2(yp, xp))
        alt = r - EARTH_RADIUS_KM
        # 7) Vis-viva orbital speed.
        speed = math.sqrt(MU_EARTH_KM3_S2 * (2/r - 1/a))
        result.append({"time": t, "lat_deg": lat, "lon_deg": lon,
                       "alt_km": alt, "speed_km_s": speed})
    return result


# Independent numerical checks; never force a longitude offset.
j2000 = datetime(2000, 1, 1, 12, tzinfo=timezone.utc)
assert abs(math.degrees(gmst_rad(j2000)) - 280.46061837) < 0.01
for e in (0.0, 0.07452088, 0.9):
    for M in (0.0, 1.0, 3.0, 5.5):
        E = solve_kepler(M, e)
        assert abs(E - e * math.sin(E) - M) < 1e-10

tracks = [propagate(el, times) for el in elements]
for el, track in zip(elements, tracks):
    print(el["label"])
    for index in (0, len(track)//2, len(track)-1):
        p = track[index]
        print(f"  {p['time'].isoformat()} | lat {p['lat_deg']:+.3f}° | "
              f"lon {p['lon_deg']:+.3f}° | alt {p['alt_km']:.2f} km | "
              f"speed {p['speed_km_s']:.4f} km/s")

if any(el["norad_id"] == 38652 for el in elements):
    ses_index = next(i for i, el in enumerate(elements) if el["norad_id"] == 38652)
    assert all(4 <= p["lon_deg"] <= 6 for p in tracks[ses_index]), (
        "SES-5 longitude outside 4–6°E: check GMST and rotation signs; do not add an offset."
    )
print("Stage 5 checks passed: GMST, Kepler residuals, SES-5 longitude (when present).")


## Stage 6 — Orbit summary

PULSAR-0 has a period of about **1.58 h** and inclination near **97.5°**, consistent with a retrograde near-polar, sun-synchronous-type orbit. Galileo 29 takes about **14.08 h**; SES-5 and QZS-2 take about **23.93 h**, approximately one sidereal day. QZS-2 has eccentricity around **0.07** and inclination around **39°**, so it is an **inclined geosynchronous orbit (IGSO)**, not a geostationary one.

Perigee and apogee are heights above the spherical Earth model; minimum and maximum speeds come from the sampled Stage 5 trajectory.

In [ ]:
# Stage 6 summarizes only trajectories already computed in Stage 5.
if not all(key in globals() for key in ("elements", "tracks", "EARTH_RADIUS_KM")):
    raise RuntimeError("Run Stages 1–5 before Stage 6.")
if not elements or len(elements) != len(tracks):
    raise ValueError("Elements and tracks are missing or have different lengths.")


def orbit_summary(el, pts):
    if not pts:
        raise ValueError(f"No propagated points for {el['label']}.")
    a, e, n = el["a_km"], el["e"], el["n_rad_s"]
    speeds = [p["speed_km_s"] for p in pts]
    return {
        "name": el["name"],
        "norad_id": el["norad_id"],
        "period_h": 2 * math.pi / n / 3600,
        "perigee_km": a * (1 - e) - EARTH_RADIUS_KM,
        "apogee_km": a * (1 + e) - EARTH_RADIUS_KM,
        "e": e,
        "i_deg": math.degrees(el["i_rad"]),
        "min_speed_km_s": min(speeds),
        "max_speed_km_s": max(speeds),
    }


def summary_table(summaries):
    # Reuse these lines in Stage 8 without recalculating anything.
    lines = ["Name | NORAD | Period [h] | Perigee [km] | Apogee [km] | e | i [deg] | Min speed [km/s] | Max speed [km/s]"]
    for s in summaries:
        lines.append(
            f"{s['name']} | {s['norad_id']} | {s['period_h']:.3f} | "
            f"{s['perigee_km']:.2f} | {s['apogee_km']:.2f} | {s['e']:.8f} | "
            f"{s['i_deg']:.4f} | {s['min_speed_km_s']:.4f} | {s['max_speed_km_s']:.4f}"
        )
    return lines


summaries = [orbit_summary(el, pts) for el, pts in zip(elements, tracks)]
print("\n".join(summary_table(summaries)))
for s in summaries:
    if s["norad_id"] == 59598:
        assert 14.0 < s["period_h"] < 14.2, "Galileo 29 period outside 14.0–14.2 hours."
        print("Galileo 29 period check passed.")


## Stage 7 — Ground tracks and orbital speed

The ground-track plot shows **PULSAR-0** making many passes that shift west, **Galileo 29** tracing a broad wave, and **SES-5** nearly a single dot because its orbital period matches Earth's rotation. **QZS-2** traces an asymmetric figure-eight: its period is also geosynchronous, but its ~39° inclination changes latitude, while its eccentric orbit spends longer near its northern apogee.

In an eccentric orbit, a satellite moves **faster near perigee** and **slower near apogee**, as described by the vis-viva equation.

In [ ]:
# Stage 7: use only the coastline file and the tracks already computed.
if not all(key in globals() for key in ("IN_DIR", "OUT_DIR", "elements", "tracks", "plt")):
    raise RuntimeError("Run Stages 1–6 before Stage 7.")
if not elements or len(elements) != len(tracks) or any(not pts for pts in tracks):
    raise ValueError("Missing or mismatched orbital tracks.")

import matplotlib.dates as mdates

def load_coastline(path):
    if not path.is_file():
        raise FileNotFoundError(f"Missing coastline input: {path}")
    segments, segment = [], []
    with path.open(encoding="utf-8") as stream:
        for line_no, line in enumerate(stream, 1):
            if not line.strip():
                if segment:
                    segments.append(segment)
                    segment = []
                continue
            fields = line.split()
            if len(fields) != 2:
                raise ValueError(f"Invalid coastline line {line_no}: expected latitude longitude")
            lat, lon = map(float, fields)  # Input is latitude FIRST.
            segment.append((lon, lat))
    if segment:
        segments.append(segment)
    if not segments:
        raise ValueError("Coastline input has no segments.")
    return segments


COLORS = ["tab:blue", "tab:orange", "tab:green", "tab:red"]
if len(elements) > len(COLORS):
    raise ValueError("Stage 7 requires one distinct color per satellite (maximum four).")
coastline = load_coastline(IN_DIR / "world_map.txt")

# Figure 1: draw coastline segments separately and ground tracks as dots.
fig, ax = plt.subplots(figsize=(13, 6))
for segment in coastline:
    ax.plot([p[0] for p in segment], [p[1] for p in segment],
            color="gray", linewidth=0.55, alpha=0.7)
for el, pts, color in zip(elements, tracks, COLORS):
    ax.scatter([p["lon_deg"] for p in pts], [p["lat_deg"] for p in pts],
               s=0.35, color=color, label=el["label"], rasterized=True)
ax.set(xlim=(-180, 180), ylim=(-90, 90), xlabel="Longitude [deg]",
       ylabel="Latitude [deg]", title="Ground tracks — common UTC window")
ax.grid(alpha=0.2)
ax.legend(markerscale=7, loc="lower left")
fig.tight_layout()
fig.savefig(OUT_DIR / "ground_tracks.png", dpi=200)
plt.show()

# Figure 2: shared UTC axis, with plain y-axis numbers on every panel.
fig, axes = plt.subplots(len(elements), 1, figsize=(12, 2.7 * len(elements)),
                         sharex=True, squeeze=False)
for ax, el, pts, color in zip(axes.flat, elements, tracks, COLORS):
    ax.plot([p["time"] for p in pts], [p["speed_km_s"] for p in pts],
            color=color, linewidth=1)
    ax.set_ylabel("Speed [km/s]")
    ax.set_title(el["label"], fontsize=10, loc="left")
    ax.ticklabel_format(axis="y", style="plain", useOffset=False)
    ax.grid(alpha=0.25)
axes[-1, 0].xaxis.set_major_formatter(mdates.DateFormatter("%d %b %H:%M", tz=timezone.utc))
axes[-1, 0].set_xlabel("Time [UTC]")
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(OUT_DIR / "velocity_timeseries.png", dpi=200)
plt.show()
print("Saved:", OUT_DIR / "ground_tracks.png", "and", OUT_DIR / "velocity_timeseries.png")


## Stage 8 — Google Earth KML and final report

Export two KML 2.2 files: Earth-fixed 3D orbital paths (altitude in **metres**) and an animated ground-track collection (one sample per **60 seconds**, split at the ±180° longitude seam). Both use the same four satellite colours as the Matplotlib plots; the report reuses the existing Stage 3 and Stage 6 tables.

In [ ]:
# Stage 8: export the already-computed data; never fabricate missing inputs.
required_names = ("elements", "tracks", "summaries", "times", "start", "end",
                  "STEP_SECONDS", "source_note", "COLORS", "OUT_DIR",
                  "summary_table", "elements_table", "to_hex", "ET")
missing = [name for name in required_names if name not in globals()]
if missing:
    raise RuntimeError(f"Run Stages 1–7 first; missing: {', '.join(missing)}")
if not elements or len(elements) != len(tracks) or len(elements) != len(summaries):
    raise ValueError("Stage 8 needs matching non-empty elements, tracks and summaries.")
if any(len(pts) != len(times) for pts in tracks):
    raise ValueError("A satellite track does not match the UTC time grid.")

KML = "http://www.opengis.net/kml/2.2"
GX = "http://www.google.com/kml/ext/2.2"
ET.register_namespace("", KML)
ET.register_namespace("gx", GX)

def tag(parent, name, value=None):
    node = ET.SubElement(parent, f"{{{KML}}}{name}")
    if value is not None:
        node.text = str(value)
    return node

def kml_color(color):
    hex_color = to_hex(color).lstrip("#")
    return "ff" + hex_color[4:6] + hex_color[2:4] + hex_color[0:2]

def new_kml(title):
    root = ET.Element(f"{{{KML}}}kml")
    doc = tag(root, "Document")
    tag(doc, "name", title)
    for index, (el, color) in enumerate(zip(elements, COLORS)):
        style = tag(doc, "Style")
        style.set("id", f"sat{index}")
        icon_style = tag(style, "IconStyle")
        tag(icon_style, "color", kml_color(color))
        tag(tag(icon_style, "Icon"), "href",
            "http://maps.google.com/mapfiles/kml/shapes/placemark_circle.png")
        line_style = tag(style, "LineStyle")
        tag(line_style, "color", kml_color(color))
        tag(line_style, "width", 2)
    return root, doc

def satellite_placemark(doc, el, index):
    pm = tag(doc, "Placemark")
    tag(pm, "name", el["label"])
    tag(pm, "styleUrl", f"#sat{index}")
    return pm

# 1) One 3D, Earth-fixed LineString per satellite.
root, doc = new_kml("Multi-Satellite Orbits | Earth-fixed")
for index, (el, pts) in enumerate(zip(elements, tracks)):
    line = tag(satellite_placemark(doc, el, index), "LineString")
    tag(line, "altitudeMode", "absolute")
    tag(line, "coordinates", " ".join(
        f"{p['lon_deg']:.6f},{p['lat_deg']:.6f},{p['alt_km']*1000:.2f}"
        for p in pts
    ))
ET.ElementTree(root).write(OUT_DIR / "orbit_tracks_ecef.kml",
                           encoding="utf-8", xml_declaration=True)

# 2) MultiTrack: minute samples; split each satellite at the dateline.
root, doc = new_kml("Multi-Satellite Ground Tracks | Animation")
for index, (el, pts) in enumerate(zip(elements, tracks)):
    pm = satellite_placemark(doc, el, index)
    multi = ET.SubElement(pm, f"{{{GX}}}MultiTrack")
    sampled = [p for p in pts if (p["time"] - start).total_seconds() % 60 == 0]
    if not sampled:
        raise ValueError(f"No 60-second samples for {el['label']}.")
    sections = [[]]
    for p in sampled:
        if sections[-1] and abs(p["lon_deg"] - sections[-1][-1]["lon_deg"]) > 180:
            sections.append([])
        sections[-1].append(p)
    for section in sections:
        track = ET.SubElement(multi, f"{{{GX}}}Track")
        tag(track, "altitudeMode", "clampToGround")
        for p in section:
            tag(track, "when", p["time"].astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"))
        for p in section:
            ET.SubElement(track, f"{{{GX}}}coord").text = (
                f"{p['lon_deg']:.6f} {p['lat_deg']:.6f} 0"
            )
ET.ElementTree(root).write(OUT_DIR / "orbit_animation.kml",
                           encoding="utf-8", xml_declaration=True)

# 3) Reuse existing tables for the reproducible text report.
report_lines = [
    "Satellite Orbit Report - built during Space Career Week 2026, GNSS Academy",
    source_note,
    f"Window (UTC): {start.isoformat()} to {end.isoformat()}; step: {STEP_SECONDS} s",
    "Model caveat: two-body Kepler model with spherical Earth and simple GMST; "
    "GP/SGP4 mean elements make results approximate, not precision predictions.",
    "", "Orbit summary:", *summary_table(summaries),
    "", "Orbital elements:", *elements_table(elements),
]
(OUT_DIR / "orbit_report.txt").write_text("\n".join(report_lines) + "\n", encoding="utf-8")

# 4) Archive exactly the requested five deliverables.
filenames = ("ground_tracks.png", "velocity_timeseries.png",
             "orbit_tracks_ecef.kml", "orbit_animation.kml", "orbit_report.txt")
for name in filenames:
    if not (OUT_DIR / name).is_file():
        raise FileNotFoundError(f"Missing Stage 7/8 output: {OUT_DIR / name}")
archive = OUT_DIR / "satellite_orbit_outputs.zip"
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for name in filenames:
        zf.write(OUT_DIR / name, arcname=name)
print("Created:", archive)
for name in filenames:
    print(f"  {name}: {(OUT_DIR / name).stat().st_size:,} bytes")
print("Archive entries:", zipfile.ZipFile(archive).namelist())


In [ ]:
# Run this separate cell in Google Colab to download the finished ZIP.
if "archive" not in globals() or not archive.is_file():
    raise RuntimeError("Run Stage 8 export before downloading.")
from google.colab import files
files.download(str(archive))


## Run all — completion checklist

- [ ] Run all notebook cells in order (Stages 1–8); upload the original input ZIP if Stage 2 requests it.
- [ ] Confirm that both Matplotlib figures appear and the Stage 8 cell lists all **five** output files.
- [ ] Run the **separate download cell** and save `satellite_orbit_outputs.zip`.
- [ ] Extract the ZIP and open `orbit_tracks_ecef.kml` in **Google Earth Pro** (File → Open) to view the 3D Earth-fixed orbital paths.
- [ ] Open `orbit_animation.kml` in **Google Earth Pro**. Expand the satellite entries if needed and use the **time slider/play control** to animate the time-tagged ground tracks. Make sure the timeline spans 10–11 November 2026.

**Reminder:** These paths are educational two-body approximations based on GP mean elements; the QZS-2 record is older than seven days at the start of the window.